# Synthetic Data Descriptive Statistics

This notebook provides a comprehensive statistical summary of the synthetic mobile money transaction data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## Load Data

In [ ]:
transactions = pd.read_csv('data/transactions_calibrated.csv')
summary_extended = pd.read_csv('data/summary_extended.csv')
user_summaries = pd.read_csv('data/user_summaries.csv')
features = pd.read_csv('data/features.csv')

print(f"Transactions: {transactions.shape}")
print(f"Summary Extended: {summary_extended.shape}")
print(f"User Summaries: {user_summaries.shape}")
print(f"Features: {features.shape}")

## Data Overview

In [ ]:
transactions.head()

In [ ]:
summary_extended.head()

## Numerical Statistics - Transactions

In [ ]:
numerical_cols = ['AMOUNT', 'FEES', 'E-LEVY', 'BAL BEFORE', 'BAL AFTER', 'hour']
transactions[numerical_cols].describe()

In [ ]:
for col in numerical_cols:
    data = transactions[col].dropna()
    print(f"\n{col}:")
    print(f"  Mean: {data.mean():.2f}")
    print(f"  Std:  {data.std():.2f}")
    print(f"  CV:   {data.std() / data.mean() * 100:.1f}%" if data.mean() != 0 else "  CV:   N/A")
    print(f"  Min:  {data.min():.2f}")
    print(f"  Max:  {data.max():.2f}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, col in zip(axes.flatten(), numerical_cols):
    transactions[col].hist(bins=50, ax=ax, edgecolor='black', alpha=0.7)
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
plt.tight_layout()
plt.savefig('notebooks/figures/numerical_distributions.png', dpi=100, bbox_inches='tight')
plt.show()

## Numerical Statistics - User Summary

In [ ]:
summary_numerical = summary_extended.select_dtypes(include=[np.number])
summary_numerical.describe()

In [ ]:
num_cols = ['loans_taken', 'final_credit_limit', 'total_transactions']
for col in num_cols:
    data = summary_extended[col].dropna()
    print(f"\n{col}:")
    print(f"  Mean: {data.mean():.2f}")
    print(f"  Std:  {data.std():.2f}")
    print(f"  CV:   {data.std() / data.mean() * 100:.1f}%" if data.mean() != 0 else "  CV:   N/A")
    print(f"  Min:  {data.min():.2f}")
    print(f"  Max:  {data.max():.2f}")

## Categorical Statistics - Transactions

In [ ]:
categorical_cols = ['TRANS. TYPE', 'is_fraud', 'fraud_type']
for col in categorical_cols:
    print(f"\n{col}:")
    print(f"  Unique values: {transactions[col].nunique()}")
    vc = transactions[col].value_counts()
    print(f"  Distribution:")
    for val, cnt in vc.items():
        print(f"    {val}: {cnt} ({cnt/len(transactions)*100:.1f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
transactions['TRANS. TYPE'].value_counts().plot(kind='bar', ax=ax, edgecolor='black')
ax.set_title('Transaction Type Distribution')
ax.set_xlabel('Transaction Type')
ax.set_ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('notebooks/figures/transaction_types.png', dpi=100, bbox_inches='tight')
plt.show()

## Categorical Statistics - User Summary

In [ ]:
cat_cols = ['credit_archetype', 'credit_risk_label']
for col in cat_cols:
    print(f"\n{col}:")
    print(f"  Unique values: {summary_extended[col].nunique()}")
    vc = summary_extended[col].value_counts()
    print(f"  Distribution:")
    for val, cnt in vc.items():
        print(f"    {val}: {cnt} ({cnt/len(summary_extended)*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

summary_extended['credit_archetype'].value_counts().plot(kind='bar', ax=axes[0], edgecolor='black')
axes[0].set_title('Credit Archetype Distribution')
axes[0].set_xlabel('Archetype')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

labels = {-1: 'Non-borrower', 0: 'Good', 1: 'Late', 2: 'Default'}
summary_extended['credit_risk_label'].map(labels).value_counts().plot(kind='bar', ax=axes[1], edgecolor='black')
axes[1].set_title('Credit Risk Label Distribution')
axes[1].set_xlabel('Risk Label')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('notebooks/figures/credit_archetype_risk.png', dpi=100, bbox_inches='tight')
plt.show()

## Temporal Distribution

In [ ]:
transactions['hour'] = pd.to_numeric(transactions['hour'], errors='coerce')
fig, ax = plt.subplots(figsize=(12, 5))
transactions['hour'].value_counts().sort_index().plot(kind='bar', ax=ax, edgecolor='black')
ax.set_title('Transactions by Hour of Day')
ax.set_xlabel('Hour')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('notebooks/figures/hour_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
weekend_counts = transactions['is_weekend'].value_counts()
print("Weekend vs Weekday:")
for val, cnt in weekend_counts.items():
    label = "Weekend" if val == True else "Weekday"
    print(f"  {label}: {cnt} ({cnt/len(transactions)*100:.1f}%)")

## Fraud Distribution

In [ ]:
fraud_counts = transactions['is_fraud'].value_counts()
print("Fraud Distribution:")
for val, cnt in fraud_counts.items():
    label = "Fraud" if val == 1 else "Legitimate"
    print(f"  {label}: {cnt} ({cnt/len(transactions)*100:.1f}%)")

print("\nFraud Types:")
fraud_types = transactions[transactions['is_fraud'] == 1]['fraud_type'].value_counts()
for ft, cnt in fraud_types.items():
    print(f"  {ft}: {cnt} ({cnt/fraud_counts[1]*100:.1f}%)")

## Key Correlations

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
num_features = features.select_dtypes(include=[np.number])
corr = num_features.corr()
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('notebooks/figures/correlation_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

## Summary Statistics Table

In [ ]:
summary_stats = pd.DataFrame({
    'Metric': [
        'Total Users',
        'Total Transactions',
        'Avg Transactions per User',
        'Unique Transaction Types',
        'Avg Transaction Amount (GHS)',
        'Std Transaction Amount (GHS)',
        'Min Transaction (GHS)',
        'Max Transaction (GHS)',
        'Non-borrowers (%)',
        'Responsible Borrowers (%)',
        'Risky Borrowers (%)',
        'Default Rate (%)'
    ],
    'Value': [
        len(summary_extended),
        len(transactions),
        len(transactions) / len(summary_extended),
        transactions['TRANS. TYPE'].nunique(),
        transactions['AMOUNT'].mean(),
        transactions['AMOUNT'].std(),
        transactions['AMOUNT'].min(),
        transactions['AMOUNT'].max(),
        (summary_extended['credit_risk_label'] == -1).sum() / len(summary_extended) * 100,
        (summary_extended['credit_archetype'] == 'responsible_borrower').sum() / len(summary_extended) * 100,
        (summary_extended['credit_archetype'] == 'risky_borrower').sum() / len(summary_extended) * 100,
        (summary_extended['credit_risk_label'] == 2).sum() / len(summary_extended) * 100
    ]
})
summary_stats